In [0]:
from pyspark.sql import functions as F
import datetime

SCHEMA = "workspace.metropulse"
SERVICE_DATE = "2026-11-04"

def bronze(name):
    return spark.table(f"{SCHEMA}.bronze_{name}")

def save(df, name):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{SCHEMA}.{name}")
    return spark.table(f"{SCHEMA}.{name}")

In [0]:
date_key = SERVICE_DATE.replace("-", "")
weekday = datetime.date.fromisoformat(SERVICE_DATE).strftime("%A").lower()

cal = bronze("calendar")
cal_dates = bronze("calendar_dates")

regular = cal.filter(
    (F.col(weekday) == "1")
    & (F.col("start_date") <= date_key)
    & (F.col("end_date") >= date_key)
).select(F.trim("service_id").alias("service_id"))

added = cal_dates.filter((F.col("date") == date_key) & (F.col("exception_type") == "1")) \
                 .select(F.trim("service_id").alias("service_id"))
removed = cal_dates.filter((F.col("date") == date_key) & (F.col("exception_type") == "2")) \
                   .select(F.trim("service_id").alias("service_id"))

active_services = regular.union(added).subtract(removed)
print(weekday, date_key)
display(active_services)

wednesday 20261104


service_id
6
413


In [0]:
silver_routes = save(
    bronze("routes").select(
        F.trim("route_id").alias("route_id"),
        F.trim("route_short_name").alias("route_short_name"),
        F.trim("route_long_name").alias("route_long_name"),
        F.col("route_type").cast("int").alias("route_type"),
    )
    .withColumn("mode", F.when(F.col("route_type") == 0, "Rail").otherwise("Bus"))
    .dropDuplicates(["route_id"]),
    "silver_routes",
)

silver_stops = save(
    bronze("stops").select(
        F.trim("stop_id").alias("stop_id"),
        F.trim("stop_name").alias("stop_name"),
        F.col("stop_lat").cast("double").alias("stop_lat"),
        F.col("stop_lon").cast("double").alias("stop_lon"),
        F.col("wheelchair_boarding").cast("int").alias("wheelchair_boarding"),
    )
    .filter(F.col("stop_lat").between(29.0, 30.6) & F.col("stop_lon").between(-96.2, -94.6))
    .dropDuplicates(["stop_id"]),
    "silver_stops",
)

In [0]:
silver_trips = save(
    bronze("trips").select(
        F.trim("trip_id").alias("trip_id"),
        F.trim("route_id").alias("route_id"),
        F.trim("service_id").alias("service_id"),
        F.trim("trip_headsign").alias("trip_headsign"),
        F.col("direction_id").cast("int").alias("direction_id"),
    )
    .join(active_services, on="service_id", how="left_semi")
    .withColumn("service_date", F.lit(SERVICE_DATE).cast("date"))
    .dropDuplicates(["trip_id"]),
    "silver_trips",
)

In [0]:
def gtfs_time_to_seconds(col_name):
    parts = F.split(F.trim(F.col(col_name)), ":")
    return (parts.getItem(0).cast("int") * 3600
            + parts.getItem(1).cast("int") * 60
            + parts.getItem(2).cast("int"))

silver_stop_times = save(
    bronze("stop_times").select(
        F.trim("trip_id").alias("trip_id"),
        F.trim("stop_id").alias("stop_id"),
        F.col("stop_sequence").cast("int").alias("stop_sequence"),
        F.trim("arrival_time").alias("arrival_time"),
        F.trim("departure_time").alias("departure_time"),
    )
    .withColumn("arrival_secs", gtfs_time_to_seconds("arrival_time"))
    .withColumn("departure_secs", gtfs_time_to_seconds("departure_time"))
    .withColumn("departure_hour", (F.floor(F.col("departure_secs") / 3600) % 24).cast("int"))
    .join(silver_trips.select("trip_id"), on="trip_id", how="left_semi")
    .dropDuplicates(["trip_id", "stop_sequence"]),
    "silver_stop_times",
)

In [0]:
silver_stop_events = save(
    silver_stop_times
    .join(silver_trips, "trip_id", "inner")
    .join(silver_routes, "route_id", "inner")
    .join(silver_stops, "stop_id", "inner"),
    "silver_stop_events",
)
display(silver_stop_events.limit(10))

stop_id,route_id,trip_id,stop_sequence,arrival_time,departure_time,arrival_secs,departure_secs,departure_hour,service_id,trip_headsign,direction_id,service_date,route_short_name,route_long_name,route_type,mode,stop_name,stop_lat,stop_lon,wheelchair_boarding
8030,002,12050909,14,3:57:51,3:57:51,14271,14271,3,6,TMC TC,0,2026-11-04,002,Bellaire,3,Bus,Bellaire Blvd @ S Kirkwood Rd,29.703528,-95.588245,0
8180,002,12050909,19,4:01:48,4:01:48,14508,14508,4,6,TMC TC,0,2026-11-04,002,Bellaire,3,Bus,Bellaire Blvd @ Turtlewood Dr,29.703834,-95.567675,0
9830,002,12050909,56,4:30:45,4:30:45,16245,16245,4,6,TMC TC,0,2026-11-04,002,Bellaire,3,Bus,Bellaire Blvd @ Alpine,29.705717,-95.453072,0
9588,002,12050909,71,4:40:17,4:40:17,16817,16817,4,6,TMC TC,0,2026-11-04,002,Bellaire,3,Bus,Holcombe Blvd @ Greenbriar Dr,29.706066,-95.412185,0
8006,002,12050910,5,4:07:23,4:07:23,14843,14843,4,6,TMC TC,0,2026-11-04,002,Bellaire,3,Bus,Bellaire Blvd @ Puerta Vista Ln,29.703512,-95.621268,0
8030,002,12050910,14,4:12:51,4:12:51,15171,15171,4,6,TMC TC,0,2026-11-04,002,Bellaire,3,Bus,Bellaire Blvd @ S Kirkwood Rd,29.703528,-95.588245,0
8180,002,12050910,19,4:16:48,4:16:48,15408,15408,4,6,TMC TC,0,2026-11-04,002,Bellaire,3,Bus,Bellaire Blvd @ Turtlewood Dr,29.703834,-95.567675,0
5974,002,12050910,41,4:32:02,4:32:02,16322,16322,4,6,TMC TC,0,2026-11-04,002,Bellaire,3,Bus,Bellaire Blvd @ Tarnef Dr,29.705229,-95.500764,0
8032,002,12050911,16,4:29:32,4:29:32,16172,16172,4,6,TMC TC,0,2026-11-04,002,Bellaire,3,Bus,Bellaire Blvd @ Baneway Dr,29.703661,-95.579179,0
11668,002,12050911,25,4:36:27,4:36:27,16587,16587,4,6,TMC TC,0,2026-11-04,002,Bellaire,3,Bus,Bellaire Blvd @ Ranchester Dr,29.704782,-95.544693,0


In [0]:
counts = {
    "bronze_trips": bronze("trips").count(),
    "silver_trips (weekday)": silver_trips.count(),
    "bronze_stop_times": bronze("stop_times").count(),
    "silver_stop_times (weekday)": silver_stop_times.count(),
    "silver_stop_events": silver_stop_events.count(),
}
for name, n in counts.items():
    print(f"{name:30} {n:>10,}")

bad_times = silver_stop_times.filter(
    F.col("arrival_secs").isNull()
    | F.col("departure_secs").isNull()
    | (F.col("departure_secs") < F.col("arrival_secs"))
).count()

assert bad_times == 0, f"{bad_times} rows have missing or backwards times"
assert counts["silver_stop_events"] == counts["silver_stop_times (weekday)"], "Rows were lost in the join"
print("\nAll quality checks passed")

bronze_trips                       21,878
silver_trips (weekday)              9,160
bronze_stop_times               1,415,650
silver_stop_times (weekday)       541,660
silver_stop_events                541,660

All quality checks passed
